Just a dev notebook to develop the algorithm. Will export the finals into a py file for execution. 

In [17]:
# Automatically reload imported modules when their source files change.
%load_ext autoreload
%autoreload 2

In [18]:
# Import the data
import numpy as np 
from utils.data_loading import load_participant_details

participant_details = load_participant_details('./raw_data/participant_details.xlsx')

participant = participant_details[0]
path = participant['filename']
data = np.loadtxt(f'./raw_data/{path}', delimiter='\t', skiprows=1, usecols=range(1, 19))

In [6]:
# Sum Humerothoracic rotation
from utils.kinematics.cumulative import calculate_arm_rotations
left_rotation, right_rotation = calculate_arm_rotations(data)
participant['left']['humerothoracic_rotation'] = left_rotation
participant['right']['humerothoracic_rotation'] = right_rotation

print(f"Left arm total rotation: {left_rotation:.2f} radians")
print(f"Right arm total rotation: {right_rotation:.2f} radians")

Left arm total rotation: 354272.47 radians
Right arm total rotation: 359999.97 radians


In [19]:
# Sum motion about each axis
from utils.kinematics.individual_axes import calculate_cumulative_axis_motion
left_axes = calculate_cumulative_axis_motion(data, 'L')
right_axes = calculate_cumulative_axis_motion(data, 'R')
print(f"Left arm cumulative axis motion (radians): {left_axes}")
print(f"Right arm cumulative axis motion (radians): {right_axes}")

C:\Users\chris\AppData\Local\Temp\ipykernel_22828\1379215796.py:3: UserWarning: Gimbal lock detected. Setting third angle to zero since it is not possible to uniquely determine all angles.
  left_axes = calculate_cumulative_axis_motion(data, 'L')


ValueError: euler_angles must not contain negative values

Getting errors about orthonormality. Tells me we might have data drift (or maybe just a bug).

Go through and see how much error is present at each timestamp. Log the values over time. 

```
I = np.eye(3)

for i, R in enumerate(matrices):
    ortho_error = np.linalg.norm(R.T @ R - I)
    det_error = np.abs(np.linalg.det(R) - 1)

    if ortho_error > 1e-6 or det_error > 1e-6:
        print(i, ortho_error, det_error)
```

| Error        | Meaning                      |
| ------------ | ---------------------------- |
| ~1e-12       | fine (floating point noise)  |
| 1e-6 to 1e-3 | mild drift / sensor noise    |
| >1e-3        | serious corruption           |
| >> 1         | not a rotation matrix at all |


Make decisions from there.

In [ ]:
from scipy.stats import describe
from utils.kinematics.general_helpers import create_rotation_matrices
matrices = create_rotation_matrices(data, "L")

# calculate variance from orthonormality
I = np.eye(3)
ortho_variance_arrays = (np.transpose(matrices, axes=(0, 2, 1)) @ matrices - I)
variances = np.linalg.norm(ortho_variance_arrays, axis=(1, 2))

summary = describe(variances)
for key, value in summary._asdict().items():
    print(f"{key:15}: {value}")

nobs           : 404800
minmax         : (1.1931257268181066e-05, 1.4005357520433481)
mean           : 0.000616359247373572
variance       : 2.1477096900688504e-05
skewness       : 119.32194121150413
kurtosis       : 27734.27044386857


In [ ]:
# import seaborn as sns
# sns.lineplot(x=range(len(variance)), y=variance)

In [ ]:
import numpy as np

# check to see if raw values are orthonormal
vals = [0.7268,	0.2742,	0.6297,	-0.3018, 0.9511, -0.0658, -0.6169, -0.1422, 0.7741, 0.9444, 0.0478, -0.3253, -0.0487, 0.9988, 0.0052, 0.3252, 0.0110, 0.9456]
left = vals[:9]
right = vals[9:]
left_matrix = np.array(left).reshape(3, 3)
right_matrix = np.array(right).reshape(3, 3)

is_left_orthonormal = np.allclose(left_matrix @ left_matrix.T, np.eye(3), atol=1e-6)
is_right_orthonormal = np.allclose(right_matrix @ right_matrix.T, np.eye(3), atol=1e-6)

print(f"Left matrix is orthonormal: {is_left_orthonormal}")
print(f"Right matrix is orthonormal: {is_right_orthonormal}")

# check to make sure relative rotation is calculated correctly
# check to make sure multiplication order is correct
# check to make sure coordinate convention is correct
# check to make sure intrinsic vs extrinsic rotations is correct
# make sure they are rotation matrices and not euler angles
# make sure left and right arms have same sequence (left not ZYX and right not XYZ)

Left matrix is orthonormal: False
Right matrix is orthonormal: False
